In [ ]:
# ============================================================
# PT -> PA Cross-Physics Mapping using DeepONet
#
# Input:
#     PT [B, 1, 501, 200]
#
# Output:
#     PA [B, 1, 501, 200]
#
# DeepONet:
#
#     Branch Net:
#         PT field -> latent coefficients b_k
#
#     Trunk Net:
#         coordinates (t, x) -> basis phi_k(t,x)
#
#     Output:
#
#         PA(t,x) = sum_k b_k * phi_k(t,x) + bias
#
# ============================================================

import os
import csv
import glob
import random
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================

DATA_ROOT = "Training dataset"

RUN_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = os.path.join(
    "deeponet_mat_dataset_results",
    f"run_{RUN_TIME}"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

print("Results will be saved to:")
print(OUT_DIR)


# ============================================================
# Dataset split
# ============================================================

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)


# ============================================================
# Training configuration
# ============================================================

EPOCHS = 1000

BATCH_SIZE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-5


# ============================================================
# DeepONet parameters
# ============================================================

IN_CHANNELS = 1
OUT_CHANNELS = 1

# Number of DeepONet latent basis functions
LATENT_DIM = 128

# Hidden size of trunk network
TRUNK_HIDDEN = 128

# Number of spatial-temporal coordinates
NT = 501
NX = 200


# ============================================================
# Random seed
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ============================================================
# Find all MAT files
# ============================================================

all_files = glob.glob(
    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),
    recursive=True
)

all_files = sorted(
    all_files
)

print(
    "Total .mat samples found:",
    len(all_files)
)

assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"


if len(all_files) != 1000:

    print(
        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Split dataset
# ============================================================

random.shuffle(
    all_files
)

n_total = len(
    all_files
)

n_train = int(
    n_total * TRAIN_RATIO
)

n_val = int(
    n_total * VAL_RATIO
)

train_files = all_files[
    :n_train
]

val_files = all_files[
    n_train:
    n_train + n_val
]

test_files = all_files[
    n_train + n_val:
]


print("\nDataset split:")

print(
    "Train samples:",
    len(train_files)
)

print(
    "Val samples  :",
    len(val_files)
)

print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# Compute normalization statistics
# Training data only
# ============================================================

def compute_statistics(
    file_list
):

    pt_sum = 0.0
    pt_sq_sum = 0.0

    pa_sum = 0.0
    pa_sq_sum = 0.0

    n_elements = 0

    print(
        "\nCalculating normalization statistics..."
    )

    for mat_path in tqdm(
        file_list,
        desc="Statistics",
        ncols=120
    ):

        with h5py.File(
            mat_path,
            "r"
        ) as f:

            PT = np.asarray(
                f["PT"],
                dtype=np.float64
            )

            PA = np.asarray(
                f["PA"],
                dtype=np.float64
            )

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"

        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"

        pt_sum += PT.sum()

        pt_sq_sum += np.square(
            PT
        ).sum()

        pa_sum += PA.sum()

        pa_sq_sum += np.square(
            PA
        ).sum()

        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (
        pt_sum
        /
        n_elements
    )

    pa_mean = (
        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (
        pt_sq_sum
        /
        n_elements
        -
        pt_mean ** 2
    )

    pa_var = (
        pa_sq_sum
        /
        n_elements
        -
        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )

    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (
        np.sqrt(
            pt_var
        )
        +
        1e-8
    )

    pa_std = (
        np.sqrt(
            pa_var
        )
        +
        1e-8
    )


    return (
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    )


# ============================================================
# Calculate normalization
# ============================================================

(
    pt_mean,
    pt_std,
    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print(
    "\nNormalization statistics:"
)

print(
    f"PT mean = {pt_mean:.6e}"
)

print(
    f"PT std  = {pt_std:.6e}"
)

print(
    f"PA mean = {pa_mean:.6e}"
)

print(
    f"PA std  = {pa_std:.6e}"
)


# ============================================================
# Save normalization statistics
# ============================================================

np.savez(
    os.path.join(
        OUT_DIR,
        "normalization_parameters.npz"
    ),
    pt_mean=pt_mean,
    pt_std=pt_std,
    pa_mean=pa_mean,
    pa_std=pa_std
)


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):

    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):

        self.file_list = file_list

        self.pt_mean = pt_mean
        self.pt_std = pt_std

        self.pa_mean = pa_mean
        self.pa_std = pa_std


    def __len__(
        self
    ):

        return len(
            self.file_list
        )


    def __getitem__(
        self,
        idx
    ):

        mat_path = self.file_list[
            idx
        ]

        with h5py.File(
            mat_path,
            "r"
        ) as f:

            PT = np.asarray(
                f["PT"],
                dtype=np.float32
            )

            PA = np.asarray(
                f["PA"],
                dtype=np.float32
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"

        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        # ====================================================
        # Normalize
        # ====================================================

        PT = (
            PT
            -
            self.pt_mean
        ) / self.pt_std

        PA = (
            PA
            -
            self.pa_mean
        ) / self.pa_std


        # ====================================================
        # [T, X]
        #
        # ->
        #
        # [C, T, X]
        # ====================================================

        PT = torch.tensor(
            PT,
            dtype=torch.float32
        ).unsqueeze(0)

        PA = torch.tensor(
            PA,
            dtype=torch.float32
        ).unsqueeze(0)


        return (
            PT,
            PA
        )


    # ========================================================
    # Denormalization
    # ========================================================

    def denormalize_pa(
        self,
        x
    ):

        return (
            x
            *
            self.pa_std
            +
            self.pa_mean
        )


    def denormalize_pt(
        self,
        x
    ):

        return (
            x
            *
            self.pt_std
            +
            self.pt_mean
        )


# ============================================================
# Create datasets
# ============================================================

train_dataset = MatHeatAcousticDataset(

    train_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


val_dataset = MatHeatAcousticDataset(

    val_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(
    iter(
        train_loader
    )
)

print(
    "\nBatch check:"
)

print(
    "PT batch shape:",
    PT_batch.shape
)

print(
    "PA batch shape:",
    PA_batch.shape
)


# ============================================================
# Branch Network
#
# PT field:
#
# [B,1,501,200]
#
# ->
#
# global latent coefficients:
#
# [B,LATENT_DIM]
#
# ============================================================

class BranchNet(nn.Module):

    def __init__(
        self,
        latent_dim=128
    ):

        super().__init__()


        self.encoder = nn.Sequential(

            # ------------------------------------------------
            # 501 x 200
            # ------------------------------------------------

            nn.Conv2d(
                1,
                32,
                kernel_size=5,
                stride=2,
                padding=2
            ),

            nn.BatchNorm2d(
                32
            ),

            nn.GELU(),


            # ------------------------------------------------
            # ~251 x 100
            # ------------------------------------------------

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(
                64
            ),

            nn.GELU(),


            # ------------------------------------------------
            # ~126 x 50
            # ------------------------------------------------

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(
                128
            ),

            nn.GELU(),


            # ------------------------------------------------
            # ~63 x 25
            # ------------------------------------------------

            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(
                256
            ),

            nn.GELU(),


            # ------------------------------------------------
            # ~32 x 13
            # ------------------------------------------------

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(
                256
            ),

            nn.GELU()
        )


        # ====================================================
        # Adaptive pooling means:
        #
        # field size does not affect FC input size
        # ====================================================

        self.pool = nn.AdaptiveAvgPool2d(
            output_size=(4, 4)
        )


        self.fc = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                256 * 4 * 4,
                512
            ),

            nn.GELU(),

            nn.Dropout(
                0.1
            ),

            nn.Linear(
                512,
                latent_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.encoder(
            x
        )

        x = self.pool(
            x
        )

        x = self.fc(
            x
        )

        return x


# ============================================================
# Trunk Network
#
# Input:
#
# coordinates:
#
# [N_points, 2]
#
# where:
#
# coordinate = (t, x)
#
# Output:
#
# [N_points, LATENT_DIM]
#
# ============================================================

class TrunkNet(nn.Module):

    def __init__(
        self,
        latent_dim=128,
        hidden_dim=128
    ):

        super().__init__()


        self.net = nn.Sequential(

            nn.Linear(
                2,
                hidden_dim
            ),

            nn.GELU(),


            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.GELU(),


            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.GELU(),


            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.GELU(),


            nn.Linear(
                hidden_dim,
                latent_dim
            )
        )


    def forward(
        self,
        coords
    ):

        return self.net(
            coords
        )


# ============================================================
# DeepONet
# ============================================================

class DeepONet2D(nn.Module):

    def __init__(
        self,
        latent_dim=128,
        trunk_hidden=128,
        nt=501,
        nx=200
    ):

        super().__init__()


        self.latent_dim = latent_dim

        self.nt = nt
        self.nx = nx


        # ====================================================
        # Branch
        # ====================================================

        self.branch = BranchNet(
            latent_dim=latent_dim
        )


        # ====================================================
        # Trunk
        # ====================================================

        self.trunk = TrunkNet(

            latent_dim=latent_dim,

            hidden_dim=trunk_hidden
        )


        # ====================================================
        # Output bias
        # ====================================================

        self.bias = nn.Parameter(
            torch.zeros(
                1
            )
        )


        # ====================================================
        # Generate normalized coordinates
        #
        # t in [-1,1]
        # x in [-1,1]
        # ====================================================

        t = torch.linspace(
            -1.0,
            1.0,
            nt
        )

        x = torch.linspace(
            -1.0,
            1.0,
            nx
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        coords = torch.stack(

            [
                tt,
                xx
            ],

            dim=-1
        )


        # ====================================================
        # [T,X,2]
        #
        # ->
        #
        # [T*X,2]
        # ====================================================

        coords = coords.reshape(
            -1,
            2
        )


        # ====================================================
        # Register as buffer
        #
        # moves automatically with model.to(device)
        # ====================================================

        self.register_buffer(
            "coords",
            coords
        )


    def forward(
        self,
        pt
    ):

        batch_size = pt.shape[
            0
        ]


        # ====================================================
        # Branch output
        #
        # [B,latent]
        # ====================================================

        branch_features = self.branch(
            pt
        )


        # ====================================================
        # Trunk output
        #
        # [T*X,latent]
        # ====================================================

        trunk_features = self.trunk(
            self.coords
        )


        # ====================================================
        # DeepONet inner product
        #
        # branch:
        # [B,L]
        #
        # trunk:
        # [N,L]
        #
        # ->
        #
        # [B,N]
        # ====================================================

        output = torch.einsum(

            "bl,nl->bn",

            branch_features,

            trunk_features
        )


        # ====================================================
        # Normalize by latent dimension
        #
        # improves initial numerical scale
        # ====================================================

        output = (
            output
            /
            np.sqrt(
                self.latent_dim
            )
        )


        output = (
            output
            +
            self.bias
        )


        # ====================================================
        # [B,T*X]
        #
        # ->
        #
        # [B,1,T,X]
        # ====================================================

        output = output.reshape(

            batch_size,

            1,

            self.nt,

            self.nx
        )


        return output


# ============================================================
# Build model
# ============================================================

model = DeepONet2D(

    latent_dim=LATENT_DIM,

    trunk_hidden=TRUNK_HIDDEN,

    nt=NT,

    nx=NX

).to(
    DEVICE
)


# ============================================================
# Model shape check
# ============================================================

with torch.no_grad():

    test_input = torch.randn(

        1,

        1,

        NT,

        NX,

        device=DEVICE
    )


    test_output = model(
        test_input
    )


print(
    "\nModel shape check:"
)

print(
    "Input shape :",
    test_input.shape
)

print(
    "Output shape:",
    test_output.shape
)


assert (
    test_output.shape
    ==
    test_input.shape
), \
    f"Shape mismatch: {test_output.shape}"


del test_input
del test_output


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# Number of parameters
# ============================================================

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    f"\nTrainable parameters: {n_params:,}"
)


# ============================================================
# Relative L2
# ============================================================

def relative_l2(
    pred,
    target
):

    numerator = torch.norm(
        pred
        -
        target
    )

    denominator = (
        torch.norm(
            target
        )
        +
        1e-8
    )

    return (
        numerator
        /
        denominator
    )


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()


    total_mse = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Forward
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # Loss
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        loss = (
            mse
            +
            0.1
            *
            rel
        )


        # ====================================================
        # Backpropagation
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        total_mse += mse.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Validation / Test
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()


    total_mse = 0.0

    total_mae = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        pred = model(
            heat
        )


        mse = F.mse_loss(

            pred,

            acoustic
        )


        mae = F.l1_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        total_mse += mse.item()

        total_mae += mae.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_mae
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Plot loss curves
# ============================================================

def plot_loss(
    log_path
):

    data = np.loadtxt(

        log_path,

        delimiter=",",

        skiprows=1
    )


    if data.ndim == 1:

        data = data[
            None,
            :
        ]


    epoch = data[
        :,
        0
    ]

    train_mse = data[
        :,
        1
    ]

    train_rel = data[
        :,
        2
    ]

    val_mse = data[
        :,
        3
    ]

    val_rel = data[
        :,
        5
    ]


    # ========================================================
    # MSE
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_mse,

        label="Train MSE"
    )


    plt.semilogy(

        epoch,

        val_mse,

        label="Validation MSE"
    )


    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "MSE"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "loss_curve.png"
        ),

        dpi=300
    )


    plt.close()


    # ========================================================
    # Relative L2
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_rel,

        label="Train Rel. L2"
    )


    plt.semilogy(

        epoch,

        val_rel,

        label="Validation Rel. L2"
    )


    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Relative L2"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "relative_l2_curve.png"
        ),

        dpi=300
    )


    plt.close()


# ============================================================
# Plot prediction
# ============================================================

@torch.no_grad()
def plot_prediction(
    model,
    dataset,
    sample_index=0
):

    model.eval()


    PT, PA = dataset[
        sample_index
    ]


    # ========================================================
    # Prediction
    # ========================================================

    PT_gpu = PT.unsqueeze(
        0
    ).to(
        DEVICE
    )


    pred = model(
        PT_gpu
    )


    pred = (
        pred
        .cpu()
        .squeeze(0)
        .squeeze(0)
        .numpy()
    )


    PT = (
        PT
        .squeeze(0)
        .numpy()
    )


    PA = (
        PA
        .squeeze(0)
        .numpy()
    )


    # ========================================================
    # Denormalize
    # ========================================================

    PT_real = dataset.denormalize_pt(
        PT
    )


    PA_real = dataset.denormalize_pa(
        PA
    )


    pred_real = dataset.denormalize_pa(
        pred
    )


    err = (
        pred_real
        -
        PA_real
    )


    # ========================================================
    # Range
    # ========================================================

    vmax = np.max(
        np.abs(
            PA_real
        )
    )


    vmax = max(
        vmax,
        1e-12
    )


    evmax = np.max(
        np.abs(
            err
        )
    )


    evmax = max(
        evmax,
        1e-12
    )


    # ========================================================
    # Figure
    # ========================================================

    plt.figure(
        figsize=(12, 3)
    )


    # --------------------------------------------------------
    # PT
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        1
    )


    plt.imshow(

        PT_real,

        aspect="auto",

        cmap="inferno"
    )


    plt.title(
        "Input PT"
    )

    plt.xlabel(
        "x"
    )

    plt.ylabel(
        "t"
    )

    plt.colorbar()


    # --------------------------------------------------------
    # GT PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        2
    )


    plt.imshow(

        PA_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "GT PA"
    )

    plt.xlabel(
        "x"
    )

    plt.ylabel(
        "t"
    )

    plt.colorbar()


    # --------------------------------------------------------
    # Predicted PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        3
    )


    plt.imshow(

        pred_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "Pred PA"
    )

    plt.xlabel(
        "x"
    )

    plt.ylabel(
        "t"
    )

    plt.colorbar()


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        4
    )


    plt.imshow(

        err,

        aspect="auto",

        cmap="seismic",

        vmin=-evmax,

        vmax=evmax
    )


    plt.title(
        "Error"
    )

    plt.xlabel(
        "x"
    )

    plt.ylabel(
        "t"
    )

    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "prediction_comparison.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # Save result
    # ========================================================

    np.savez(

        os.path.join(
            OUT_DIR,
            "prediction_result.npz"
        ),

        PT=PT_real,

        PA=PA_real,

        PA_pred=pred_real,

        error=err
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS
)


# ============================================================
# Training log
# ============================================================

log_path = os.path.join(

    OUT_DIR,

    "training_log.csv"
)


with open(
    log_path,
    "w",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow(
        [
            "epoch",
            "train_mse",
            "train_rel_l2",
            "val_mse",
            "val_mae",
            "val_rel_l2",
            "lr"
        ]
    )


# ============================================================
# Best model
# ============================================================

best_val = float(
    "inf"
)


best_model_path = os.path.join(

    OUT_DIR,

    "best_deeponet.pt"
)


# ============================================================
# Training
# ============================================================

epoch_bar = tqdm(

    range(
        1,
        EPOCHS + 1
    ),

    desc="Training",

    ncols=120
)


for epoch in epoch_bar:


    # ========================================================
    # Training
    # ========================================================

    train_mse, train_rel = train_one_epoch(

        model,

        train_loader,

        optimizer
    )


    # ========================================================
    # Validation
    # ========================================================

    (
        val_mse,
        val_mae,
        val_rel

    ) = evaluate(

        model,

        val_loader
    )


    # ========================================================
    # Learning rate
    # ========================================================

    scheduler.step()


    lr_now = optimizer.param_groups[
        0
    ]["lr"]


    # ========================================================
    # Save log
    # ========================================================

    with open(
        log_path,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(
            f
        )


        writer.writerow(
            [
                epoch,
                train_mse,
                train_rel,
                val_mse,
                val_mae,
                val_rel,
                lr_now
            ]
        )


    # ========================================================
    # Save best model
    # ========================================================

    if val_rel < best_val:

        best_val = val_rel


        torch.save(

            model.state_dict(),

            best_model_path
        )


    # ========================================================
    # Print
    # ========================================================

    if (
        epoch == 1
        or
        epoch % 10 == 0
    ):

        print(

            f"\nEpoch {epoch:04d} | "

            f"Train MSE {train_mse:.4e} | "

            f"Train Rel {train_rel:.4e} | "

            f"Val MSE {val_mse:.4e} | "

            f"Val MAE {val_mae:.4e} | "

            f"Val Rel {val_rel:.4e}"
        )


    epoch_bar.set_postfix(

        train_mse=f"{train_mse:.2e}",

        val_mse=f"{val_mse:.2e}",

        rel=f"{val_rel:.2e}",

        lr=f"{lr_now:.1e}"
    )


# ============================================================
# Load best model
# ============================================================

print(
    "\nLoading best model..."
)


model.load_state_dict(

    torch.load(

        best_model_path,

        map_location=DEVICE,

        weights_only=True
    )
)


# ============================================================
# Final test
# ============================================================

(
    test_mse,
    test_mae,
    test_rel

) = evaluate(

    model,

    test_loader
)


print(
    "\nFinal Test Results"
)


print(
    f"Test MSE     : {test_mse:.6e}"
)


print(
    f"Test MAE     : {test_mae:.6e}"
)


print(
    f"Test Rel L2  : {test_rel:.6e}"
)


# ============================================================
# Save test metrics
# ============================================================

with open(

    os.path.join(
        OUT_DIR,
        "test_results.txt"
    ),

    "w"

) as f:


    f.write(
        "Final Test Results\n"
    )


    f.write(
        f"Test MSE     : {test_mse:.6e}\n"
    )


    f.write(
        f"Test MAE     : {test_mae:.6e}\n"
    )


    f.write(
        f"Test Rel L2  : {test_rel:.6e}\n"
    )


# ============================================================
# Plot
# ============================================================

plot_loss(
    log_path
)


plot_prediction(

    model,

    test_dataset,

    sample_index=0
)


# ============================================================
# Finish
# ============================================================

print(
    f"\nBest validation Rel L2: "
    f"{best_val:.6e}"
)


print(
    f"Best model saved to: "
    f"{best_model_path}"
)


print(
    f"\nAll results saved to: "
    f"{OUT_DIR}"
)

Results will be saved to:
deeponet_mat_dataset_results/run_20260825_130635
Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics: 100%|█████████████████████████████████████████████████████████████████████| 640/640 [00:59<00:00, 10.75it/s]



Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([4, 1, 501, 200])
PA batch shape: torch.Size([4, 1, 501, 200])

Model shape check:
Input shape : torch.Size([1, 1, 501, 200])
Output shape: torch.Size([1, 1, 501, 200])

Trainable parameters: 3,209,665


Training:   0%|     | 1/1000 [01:04<17:49:12, 64.22s/it, lr=1.0e-03, rel=1.07e+00, train_mse=9.34e-01, val_mse=9.42e-01]


Epoch 0001 | Train MSE 9.3419e-01 | Train Rel 9.5732e-01 | Val MSE 9.4206e-01 | Val MAE 2.8815e-01 | Val Rel 1.0657e+00


...

Training:  13%|▏| 130/1000 [2:58:46<19:45:56, 81.79s/it, lr=9.6e-04, rel=7.73e-01, train_mse=3.42e-01, val_mse=3.50e-01]


Epoch 0130 | Train MSE 3.4207e-01 | Train Rel 6.3348e-01 | Val MSE 3.4951e-01 | Val MAE 1.7331e-01 | Val Rel 7.7314e-01


Training:  14%|▏| 140/1000 [3:12:01<18:59:23, 79.49s/it, lr=9.5e-04, rel=8.92e-01, train_mse=3.05e-01, val_mse=3.59e-01]


Epoch 0140 | Train MSE 3.0493e-01 | Train Rel 6.1571e-01 | Val MSE 3.5898e-01 | Val MAE 1.8053e-01 | Val Rel 8.9160e-01


Training:  14%|▏| 142/1000 [3:14:40<19:00:51, 79.78s/it, lr=9.5e-04, rel=8.40e-01, train_mse=3.22e-01, val_mse=4.32e-01]